# Motif Phylogeny

Fingerprint quantum algorithms by ZX motif profiles, cluster into a phylogenetic tree, and uncover structural relationships across algorithm families.

Replaces the batch script `discover_phylogeny.py` with an interactive walkthrough.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist

from zx_motifs.algorithms.registry import ALGORITHM_FAMILY_MAP, REGISTRY
from zx_motifs.pipeline.converter import SimplificationLevel
from zx_motifs.pipeline.fingerprint import build_corpus, discover_motifs, build_fingerprint_matrix
from zx_motifs.pipeline.matcher import find_motif_in_graph
from zx_motifs.pipeline.decomposer import decompose_across_corpus

%matplotlib inline

## 1. Build Corpus

Convert all 78 registered algorithms to ZX diagrams at every simplification level, then featurize into labeled NetworkX graphs.

In [ ]:
corpus = build_corpus(max_qubits=5)
n_instances = len({name for name, _ in corpus})
n_graphs = len(corpus)
print(f"{n_instances} algorithm instances, {n_graphs} total graphs")

## 2. Discover Motifs

Combine hand-crafted motif patterns with bottom-up subgraph enumeration and neighbourhood extraction. Deduplicate via WL hash + VF2 confirmation.

In [ ]:
motifs = discover_motifs(corpus)
print(f"{len(motifs)} total motifs after deduplication")

## 3. Fingerprint Matrix

Count motif occurrences per algorithm at `spider_fused` level. L1-normalise each row to get frequency vectors.

In [ ]:
counts_df, freq_df = build_fingerprint_matrix(corpus, motifs)
print(f"Fingerprint matrix: {counts_df.shape[0]} instances × {counts_df.shape[1]} motifs")

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(freq_df.T, cmap="YlOrRd", ax=ax, cbar_kws={"label": "Frequency"})
ax.set_title("Motif Fingerprint Heatmap")
ax.set_xlabel("Algorithm Instance")
ax.set_ylabel("Motif")
plt.xticks(fontsize=5, rotation=90)
plt.yticks(fontsize=6)
plt.tight_layout()
plt.show()

## 4. Phylogenetic Clustering

Hierarchical clustering on cosine distance reveals which algorithms are structurally related regardless of their intended purpose.

In [ ]:
FAMILY_COLOURS = {
    "oracle": "#e41a1c", "entanglement": "#377eb8", "error_correction": "#4daf4a",
    "distillation": "#984ea3", "protocol": "#ff7f00", "variational": "#a65628",
    "simulation": "#f781bf", "transform": "#999999", "arithmetic": "#66c2a5",
    "machine_learning": "#e6ab02", "linear_algebra": "#1b9e77", "cryptography": "#d95f02",
    "sampling": "#7570b3", "error_mitigation": "#e7298a", "topological": "#66a61e",
    "metrology": "#e6ab02", "differential_equations": "#a6761d", "tda": "#666666",
    "communication": "#1f78b4",
}

def _get_family(instance_name):
    base = instance_name.rsplit("_q", 1)[0]
    return ALGORITHM_FAMILY_MAP.get(base, "unknown")

mat = freq_df.fillna(0).values
row_norms = np.linalg.norm(mat, axis=1, keepdims=True)
mat = np.where(row_norms == 0, 1e-10, mat)
dist = pdist(mat, metric="cosine")
dist = np.nan_to_num(dist, nan=1.0)
Z = linkage(dist, method="average")

fig, ax = plt.subplots(figsize=(16, max(8, len(freq_df) * 0.25)))
labels = list(freq_df.index)
dendro = dendrogram(Z, labels=labels, orientation="left", leaf_font_size=7, ax=ax)

for lbl in ax.get_yticklabels():
    fam = _get_family(lbl.get_text())
    lbl.set_color(FAMILY_COLOURS.get(fam, "#333333"))

ax.set_title("Quantum Algorithm Phylogeny (ZX Motif Fingerprints)", fontsize=14)
ax.set_xlabel("Cosine Distance")

from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=c, label=f) for f, c in FAMILY_COLOURS.items()]
ax.legend(handles=legend_handles, loc="upper right", fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## 5. PCA Scatter

2D projection of fingerprint vectors, coloured by algorithm family.

In [ ]:
mat_c = mat - mat.mean(axis=0)
U, S, Vt = np.linalg.svd(mat_c, full_matrices=False)
coords = U[:, :2] * S[:2]
var_explained = (S[:2] ** 2) / max((S**2).sum(), 1e-10)

fig, ax = plt.subplots(figsize=(10, 8))
instances = list(freq_df.index)
families = [_get_family(inst) for inst in instances]

for fam in sorted(set(families)):
    idx = [i for i, f in enumerate(families) if f == fam]
    ax.scatter(coords[idx, 0], coords[idx, 1],
               c=FAMILY_COLOURS.get(fam, "#333333"), label=fam, alpha=0.7, s=40)

ax.set_xlabel(f"PC1 ({var_explained[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({var_explained[1]:.1%} variance)")
ax.set_title("PCA of ZX Motif Fingerprints")
ax.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()

## 6. Universality Spectrum

Classify each motif as universal (≥80% of families), common (≥40%), or family-specific.

In [ ]:
family_of = {inst: _get_family(inst) for inst in counts_df.index}
all_families = sorted(set(family_of.values()))
n_families = len(all_families)

motif_stats = []
for motif_id in counts_df.columns:
    col = counts_df[motif_id]
    present = col[col > 0].index.tolist()
    present_fams = set(family_of[inst] for inst in present)
    n_fam = len(present_fams)
    if n_fam >= n_families * 0.8:
        cat = "universal"
    elif n_fam >= n_families * 0.4:
        cat = "common"
    elif n_fam > 0:
        cat = "specific"
    else:
        cat = "unused"
    motif_stats.append({"motif_id": motif_id, "n_families": n_fam, "category": cat})

df_stats = pd.DataFrame(motif_stats).sort_values("n_families", ascending=False)
cat_colours = {"universal": "#2ca02c", "common": "#1f77b4", "specific": "#ff7f0e", "unused": "#cccccc"}

fig, ax = plt.subplots(figsize=(max(10, len(df_stats) * 0.4), 6))
bar_colors = [cat_colours[c] for c in df_stats["category"]]
ax.bar(range(len(df_stats)), df_stats["n_families"], color=bar_colors)
ax.set_xticks(range(len(df_stats)))
ax.set_xticklabels(df_stats["motif_id"], rotation=90, fontsize=6)
ax.set_ylabel("Number of Families with Motif")
ax.set_title("Motif Universality Spectrum")
ax.axhline(y=n_families * 0.8, color="#2ca02c", linestyle="--", alpha=0.5, label="universal")
ax.axhline(y=n_families * 0.4, color="#1f77b4", linestyle="--", alpha=0.5, label="common")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

for cat in ["universal", "common", "specific", "unused"]:
    n = sum(1 for m in motif_stats if m["category"] == cat)
    print(f"  {cat}: {n}")

## 7. Cross-Level Survival

Which motifs survive aggressive ZX simplification? Motifs persisting to `full_reduce` are **irreducible** — they encode non-Clifford structure that cannot be simplified away.

In [ ]:
levels = [lvl.value for lvl in SimplificationLevel]

# One representative per algorithm (smallest qubit count)
seen_bases = {}
for name, level in corpus:
    base = name.rsplit("_q", 1)[0]
    if base not in seen_bases or name < seen_bases[base]:
        seen_bases[base] = name
representatives = sorted(set(seen_bases.values()))

motif_ids = [mp.motif_id for mp in motifs]
survival = np.zeros((len(motifs), len(levels)), dtype=float)

for j, level in enumerate(levels):
    for rep in representatives:
        key = (rep, level)
        if key not in corpus:
            continue
        host = corpus[key]
        for i, mp in enumerate(motifs):
            matches = find_motif_in_graph(mp.graph, host, max_matches=1)
            if len(matches) > 0:
                survival[i, j] += 1

n_reps = len(representatives)
if n_reps > 0:
    survival /= n_reps

fig, ax = plt.subplots(figsize=(10, max(6, len(motifs) * 0.35)))
sns.heatmap(survival, xticklabels=levels, yticklabels=motif_ids,
            cmap="YlOrRd", vmin=0, vmax=1, ax=ax,
            cbar_kws={"label": "Fraction of algorithms with motif"})
ax.set_title("Motif Survival Across Simplification Levels")
ax.set_xlabel("Simplification Level")
ax.set_ylabel("Motif")
plt.yticks(fontsize=6)
plt.tight_layout()
plt.show()

# Identify irreducible motifs
fr_idx = levels.index("full_reduce") if "full_reduce" in levels else -1
survivors = [motif_ids[i] for i in range(len(motif_ids))
             if fr_idx >= 0 and survival[i, fr_idx] > 0]
print(f"\nMotifs surviving full_reduce: {survivors}")

## 8. Coverage Analysis

Greedy set-cover decomposition shows what fraction of each algorithm's ZX structure can be explained by the motif library.

In [ ]:
results = decompose_across_corpus(corpus, motifs, target_level="spider_fused")

coverage_data = []
for algo_name, dr in results.items():
    base = algo_name.rsplit("_q", 1)[0]
    fam = ALGORITHM_FAMILY_MAP.get(base, "unknown")
    coverage_data.append({"algorithm": algo_name, "family": fam, "coverage": dr.coverage_ratio})

cov_df = pd.DataFrame(coverage_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
if not cov_df.empty:
    families_present = sorted(cov_df["family"].unique())
    box_data = [cov_df[cov_df["family"] == f]["coverage"].values for f in families_present]
    bp = axes[0].boxplot(box_data, labels=families_present, patch_artist=True)
    for patch, fam in zip(bp["boxes"], families_present):
        patch.set_facecolor(FAMILY_COLOURS.get(fam, "#999999"))
    axes[0].set_ylabel("Coverage Ratio")
    axes[0].set_title("Motif Coverage by Algorithm Family")
    axes[0].tick_params(axis="x", rotation=45)

    axes[1].hist(cov_df["coverage"], bins=20, color="#1f77b4", edgecolor="white")
    axes[1].set_xlabel("Coverage Ratio")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Coverage Distribution")

plt.tight_layout()
plt.show()

print(f"\nOverall mean coverage: {cov_df['coverage'].mean():.1%}")